In [ ]:
import os
import tempfile
from typing import List, Optional
import logging
from datetime import datetime

import duckdb
from google.cloud import storage
import boto3
from botocore.client import Config
from botocore.exceptions import ClientError

class CloudStorageTransfer:
    def __init__(
        self, 
        gcs_bucket_name: str,
        s3_bucket_name: str,
        s3_endpoint_url: str,
        s3_access_key: str,
        s3_secret_key: str,
        s3_region_name: str = 'us-east-1',
        ssl_verify: bool = True
    ):
        """
        Initializes the CloudStorageTransfer with GCS and S3 clients, and sets up a DuckDB connection.
        
        Args:
            gcs_bucket_name: Name of the Google Cloud Storage bucket
            s3_bucket_name: Name of the S3 bucket
            s3_endpoint_url: S3 endpoint URL
            s3_access_key: S3 access key
            s3_secret_key: S3 secret key
            s3_region_name: S3 region name (default: 'us-east-1')
            ssl_verify: Whether to verify SSL certificates (default: True)
        """
        # Set up logging
        self.logger = logging.getLogger(__name__)
        self.logger.setLevel(logging.INFO)
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
        self.logger.addHandler(handler)

        self.gcs_bucket_name = gcs_bucket_name
        self.s3_bucket_name = s3_bucket_name
        
        # Initialize GCS client
        try:
            self.gcs_client = storage.Client()
            self.gcs_bucket = self.gcs_client.bucket(gcs_bucket_name)
        except Exception as e:
            self.logger.error(f"Failed to initialize GCS client: {str(e)}")
            raise
        
        # Initialize S3 client with proper SSL verification
        try:
            self.s3_client = boto3.client(
                's3',
                aws_access_key_id=s3_access_key,
                aws_secret_access_key=s3_secret_key,
                endpoint_url=s3_endpoint_url,
                region_name=s3_region_name,
                config=Config(signature_version='s3v4'),
                verify=ssl_verify
            )
        except Exception as e:
            self.logger.error(f"Failed to initialize S3 client: {str(e)}")
            raise
        
        # Initialize DuckDB connection
        self.duckdb_conn = duckdb.connect()

    def add_encryption_key(self, key_name: str, encryption_key: str) -> None:
        """
        Adds an encryption key to the DuckDB session securely.
        """
        try:
            # Use parameterized query to prevent SQL injection
            self.duckdb_conn.execute(
                "PRAGMA add_parquet_key(?, ?);",
                [key_name, encryption_key]
            )
            self.logger.info(f"Successfully added encryption key: {key_name}")
        except Exception as e:
            self.logger.error(f"Failed to add encryption key: {str(e)}")
            raise

    def list_files_in_gcs(self, gcs_folder_path: str) -> List[str]:
        """
        Lists Parquet files in a GCS bucket folder.
        """
        try:
            prefix = gcs_folder_path.rstrip('/') + '/'
            blobs = self.gcs_bucket.list_blobs(prefix=prefix)
            files = [blob.name for blob in blobs if blob.name.endswith('.parquet')]
            self.logger.info(f"Found {len(files)} Parquet files in {gcs_folder_path}")
            return files
        except Exception as e:
            self.logger.error(f"Failed to list files in GCS: {str(e)}")
            raise

    def process_files(
        self, 
        gcs_folder_path: str, 
        s3_prefix_export: str, 
        encryption_key_name: str,
        retry_attempts: int = 3
    ) -> None:
        """
        Processes files between GCS and S3 using temporary storage with retry logic.
        """
        files = self.list_files_in_gcs(gcs_folder_path)
        
        if not files:
            self.logger.warning(f"No Parquet files found in {gcs_folder_path}")
            return

        with tempfile.TemporaryDirectory() as temp_dir:
            for file_path in files:
                retry_count = 0
                while retry_count < retry_attempts:
                    try:
                        self._process_single_file(
                            file_path,
                            temp_dir,
                            s3_prefix_export,
                            encryption_key_name
                        )
                        break
                    except Exception as e:
                        retry_count += 1
                        if retry_count == retry_attempts:
                            self.logger.error(f"Failed to process {file_path} after {retry_attempts} attempts: {str(e)}")
                        else:
                            self.logger.warning(f"Retry {retry_count}/{retry_attempts} for {file_path}")

    def _process_single_file(
        self, 
        file_path: str,
        temp_dir: str,
        s3_prefix_export: str,
        encryption_key_name: str
    ) -> None:
        """
        Processes a single file with proper error handling and cleanup.
        """
        temp_input_path = os.path.join(temp_dir, f"input_{datetime.now().timestamp()}.parquet")
        temp_output_path = os.path.join(temp_dir, f"encrypted_{datetime.now().timestamp()}.parquet")
        
        try:
            # Download from GCS
            blob = self.gcs_bucket.blob(file_path)
            blob.download_to_filename(temp_input_path)
            self.logger.info(f"Downloaded {file_path} to temporary storage")
            
            # Encrypt using DuckDB
            self.duckdb_conn.execute(
                """
                COPY (
                    SELECT * FROM read_parquet(?)
                ) TO ? (
                    FORMAT 'parquet',
                    ENCRYPTION_CONFIG {footer_key: ?}
                );
                """,
                [temp_input_path, temp_output_path, encryption_key_name]
            )
            self.logger.info(f"Encrypted {file_path}")
            
            # Upload to S3
            s3_key = f"{s3_prefix_export}/{os.path.basename(file_path)}"
            extra_args = {
                'ContentType': 'application/x-parquet',
                'ACL': 'private'
            }
            
            with open(temp_output_path, 'rb') as f:
                self.s3_client.upload_fileobj(
                    f,
                    self.s3_bucket_name,
                    s3_key,
                    ExtraArgs=extra_args
                )
            
            self.logger.info(f"Successfully uploaded: {file_path} to s3://{self.s3_bucket_name}/{s3_key}")
            
        finally:
            # Clean up temporary files
            for temp_file in [temp_input_path, temp_output_path]:
                if os.path.exists(temp_file):
                    os.remove(temp_file)

    def test_s3_connection(self) -> bool:
        """
        Tests the S3 connection by listing the bucket contents.
        """
        try:
            self.s3_client.head_bucket(Bucket=self.s3_bucket_name)
            self.logger.info("Successfully connected to S3")
            return True
        except ClientError as e:
            error_code = e.response.get('Error', {}).get('Code', 'Unknown')
            self.logger.error(f"S3 connection test failed with error code {error_code}: {str(e)}")
            return False
        except Exception as e:
            self.logger.error(f"Unexpected error testing S3 connection: {str(e)}")
            return False

: 